## Forward Pass

We have $Q \in \mathcal{R}^{N \times d}$, $K \in \mathcal{R}^{M \times d}$, and $V \in \mathcal{R}^{M \times d}$.

We want to compute 
$$
\text{Attn}(Q, K, V) = \text{Softmax}(\tau QK^T)V
$$

Let's break it down 

$$
S = QK^T \\
P = \text{Softmax}(\tau S) \\
O = PV
$$

We want to compute the above in online fashion, where only the blocks of Q, K, and V are available at a time. This might be due to memory constraints of GPU (e.g., FlashAttention) or due to the harware memoery constraints (e.g., Ring Attention). 

Now, note that the full computation can be written as 

$$
S_{ij} = Q_i^TK_j \\ 

P_{ij} = \frac{\exp (\tau S_{ij})}{\sum_{j} \exp (\tau S_{ij})} \\

O_{i, :} = \sum_{j}P_{i, j} \times V_{j, :}
$$

Thus, we want to compute 

$$
O_{i, :} = \frac{\sum_{j}\exp (\tau Q_i^TK_j)  \times V_{j, :}}{\sum_{j} \exp (\tau Q_i^TK_j)}
$$

To avoid overflow resulting from large exponents, we normalize the expoenentials by subtracting the maximum of scores from their exponents. 

$$
m_i = \max_{j}Q_i^TK_j \\

O_{i, :} = \frac{\sum_{j}\exp (\tau Q_i^TK_j - m_i)  \times V_{j, :}}{\sum_{j} \exp (\tau Q_i^TK_j  - m_i)}
$$


Let's denote the block of queries by $B_q$, and the block of keys and values as $B_k$.
Thus, we have $Q_{B_q} \in \mathcal{R}^{B_q \times d}$, and $K_{B_k} \in \mathcal{R}^{B_k \times d}$, and $V \in \mathcal{R}^{B_k \times d}$, where $B_q < N$ and $B_k < M$.
To do the above compuation, when only blocks of $Q, K$, and $V$ are available, we need to break it down in blocks. 


For convenience, imagine $B_q$ and $B_k$ to be $1$. For a given query, when we have read the $t$-th key and value, we can upadte the numerator, denominator, and m_i as follows:

$$
m_i^t = \max (m_i^{t-1}, Q_i^TK_t) \in \mathcal{R}\\ 

n_i^t = n_i^{t-1} \times \exp (m_i^{t-1} - m_i^t) + \exp (\tau Q_i^TK_t - m_i^{t}) \times V_t \in \mathcal{R}^d \\ 

d_i^t = d_i^{t-1} \times \exp (m_i^{t-1} - m_i^t) + \exp (\tau Q_i^TK_t - m_i^{t}) \in \mathcal{R}^d \\

$$

Note that the correction term $\exp (m_i^{t-1} - m_i^t)$ accounts for the update in the maximum for normalizing the exponents. 

At the end of the $M$-th reading of the keys, we will have

$$
O_{i, :} = \frac{n_i^{M}}{d_i^{M}}
$$

Thus, we can compute the attention in an online fashion, where we only need to keep the blocks of Query, Keys, and Values in memory and update the output matrix iteratively.



## Backward Pass

Let's simplify the above notations a bit. At the end of the above forward computation, we have access to the sum in the denominator, $d_i = d^{M}_i$, and the $m_i = \max_j Q_i^TK_j$. 

During the backward pass, we are interested in computing $\frac{\partial L}{\partial Q}$, $\frac{\partial L}{\partial K}$, $\frac{\partial L}{\partial V}$, given $\frac{\partial L}{\partial O}$, where $L \in \mathcal{R}$ is the loss function. For simplicity, we will avoid typing $\partial L$ so that $\partial O = \frac{\partial L}{\partial O}$ and so on. 

Note,
$$
\partial O \in \mathcal{R}^{N \times d}, \quad \partial Q \in \mathcal{R}^{N \times d}, \quad  \partial K \in \mathcal{R}^{M \times d}, \quad  \partial V \in \mathcal{R}^{M \times d}
$$

$$
O = PV  \in \mathcal{R}^{N \times d}\\

\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \qquad \partial P = \partial O V^T \in \mathcal{R}^{N \times M} \\
$$

Since each $P_{i, :}$ is a result of the softmax on the $i$-th row of $S$, while taking the derivative, we need to treat them together. 

Specifically,

$$
\mathbf{y} = \text{Softmax}(\mathbf{x}) \in \mathcal{R}^d \\

\partial \mathbf{x} = (\text{diag}(\mathbf{y}) - \mathbf{y}\mathbf{y}^T) \times \partial \mathbf{y}
$$

Above follows from the following observation - 

$$
\frac{\partial \mathbf{y}_{i}}{\partial \mathbf{x}_j} = \sigma_i \sigma_j, \quad \text{if i} \neq {j} \\
= \sigma_i (1 - \sigma_i), \text{otherwise}
$$

Applying it to the $i$-th row of $P$, we have

$$
\partial S_{i, :} = (\text{diag}(\mathbf{P_{i, :}}) - \mathbf{P_{i, :}}\mathbf{P_{i, :}}^T) \times \partial \mathbf{P_{i, :}} \in \mathcal{R}^{M},
$$

where I have assumed $\mathbf{P_{i, :}} \in \mathcal{R}^M$, a column vector even though it represents a row in the matrix $P$ 

Given $\partial S_{ij}$, we can proceed in the similar fashion to compute $\partial Q$ and $\partial K$ since $S = QK^T$ is the matrix multiplication.

$$
\partial Q = \partial S K \in \mathcal{R}^{N \times d}, \qquad \partial K^T = Q^T \partial S \in \mathcal{R}^{d \times M}

$$


If you are an expert in matrix calculus, above is easy. If you are not, checkout the appendix on how to arrive at the above formulas. 

### Online pass





## Appendix 

Here we will build the intuition to perform the following,

$$
O = PV  \in \mathcal{R}^{N \times d}\\

\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \qquad \partial P = \partial O V^T \in \mathcal{R}^{N \times M} \\
$$

### Computing $\partial V$

Let's focus on the $i$-th row of $V \in \mathcal{R}^{M \times d}$.

In the forward computation we have, using the row form of matrix multiplication,  
$$
O_{i, :} = \sum_j P_{ij} \times V_{j, :}
$$

Let's see how much will O_{i, :} change, if we perturn $V_{j, :}$ by $\delta V_{j, :}$

$$
\delta O_{i, :} = ..
$$

Thus, each $V_{j, :}$ contributes to $O_{i, :}$ through $P_{ij}$. As a result, during the backward pass, to compute $\partial V_{j, :}$.

$$
\langle \frac{\partial L}{\partial O}, \delta O \rangle_F = trace(\frac{\partial L}{\partial O}^T \delta O)
$$